# Day 10 · Qwen3-VL-4B grounding · Kaggle GPU

Resume E5 from the completed E0–E4 results in Kaggle Version 3. Attach that version's output as Notebook Input, and keep the road-damage dataset input attached. E3's trained adapter and projector are restored; the notebook runs E5, scores it, and exports evidence. No test score is entered manually.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
REPO = "https://github.com/vminhquan/K4-Track-4-Day10-VoMinhQuan-2A202602429-VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print("Working in", Path.cwd())

PREVIOUS = Path("/kaggle/input/notebooks/minhquanvo/notebook3f3cc8f9bf/day10")
if not (PREVIOUS / "report/evidence/e4_data_0.1.json").is_file():
    raise FileNotFoundError("Attach completed Version 3 output as Notebook Input before running")
Path("report/evidence").mkdir(parents=True, exist_ok=True)
for source in (PREVIOUS / "report/evidence").glob("*.json"):
    shutil.copy2(source, Path("report/evidence") / source.name)
Path("work").mkdir(exist_ok=True)
shutil.copytree(PREVIOUS / "work/e3", Path("work/e3"), dirs_exist_ok=True)
for source in (PREVIOUS / "work").glob("*_test.jsonl"):
    shutil.copy2(source, Path("work") / source.name)
for name in ("e1", "e2", "e4_data_0.1"):
    source = PREVIOUS / "work" / name / "run.json"
    if source.is_file():
        target = Path("work") / name / "run.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
print("Restored E0–E4 evidence, raw predictions, and E3 checkpoint from Version 3")

def run(*args):
    command = [sys.executable, *map(str, args)]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


## 0. Install and validate the examiner tools

Run after enabling GPU. Package installation may require a kernel restart on Kaggle. Rerun this notebook from the first cell after a restart; `data/` and completed checkpoints remain in `/kaggle/working` for the current session.


In [ ]:
run("-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "pytest")
# Kaggle ships torchao 0.10; PEFT 0.21.2 rejects it while injecting ViT LoRA.
# This workflow uses bitsandbytes NF4, not torchao quantization.
run("-m", "pip", "uninstall", "-y", "torchao")
run("-m", "pytest", "-q")
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("-m", "unittest", "-v", "test_student_workflow.py")


## 1. Prepare the fixed session split

Attach `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` via **Add Input**. The examiner's script finds its `images/` and `labels-YOLO/` folder under `/kaggle/input`.


In [ ]:
if not Path("data/splits/meta.json").exists():
    run("scripts/prepare_data.py", "--source", "/kaggle/input")
run("lab_workflow.py", "check-data")
version = json.loads(Path("data/splits/meta.json").read_text())["dataset_version"]
for name in ("e0", "e1", "e2", "e3", "e4_data_0.1", "e4_data_1.0"):
    evidence = Path("report/evidence") / f"{name}.json"
    assert evidence.is_file(), name
    assert json.loads(evidence.read_text())["dataset_version"] == version, name
run("lab_workflow.py", "gt-figures")
run("lab_workflow.py", "preflight")


## 2. E0 zero-shot on the full 311-image test split

Save the raw predictions in `work/e0_test.jsonl`; the scoring script writes the evidence JSON.


In [ ]:
assert Path("report/evidence/e0.json").is_file()
print("E0 restored from Version 3")


## 3. E1: LoRA on the LLM only

A fresh Python process runs each stage, so VRAM is released before evaluation. Each SFT run saves an adapter and `run.json` every 100 steps. If Kaggle reports OOM, use `--batch-size 1 --grad-accum 8` in the train call; effective batch remains 8.


In [ ]:
assert Path("report/evidence/e1.json").is_file()
print("E1 restored from Version 3")


## 4. E2: LLM LoRA + full float32 projector

The projector is saved separately as `work/e2/projector.pt` and reloaded for inference.


In [ ]:
assert Path("report/evidence/e2.json").is_file()
print("E2 restored from Version 3")


## 5. E3: E2 + ViT LoRA

Expected LoRA target counts: E1/E2 = 252, E3 = 348. The code stops if the model module tree differs.


In [ ]:
assert all(Path(p).exists() for p in (
    "report/evidence/e3.json", "work/e3/run.json", "work/e3/adapter",
    "work/e3/projector.pt", "work/e3_test.jsonl"))
print("E3 restored from Version 3")
run("lab_workflow.py", "results")


## 6. E4: clean ablation of train data size

Pick the best E1–E3 by test F1. The 100% condition reuses its existing checkpoint and reruns inference/scoring; the 10% condition retrains from the same base model with every other setting unchanged. The fixed 10% sample uses seed 42. Both evidence files are generated by `score.py`.


In [ ]:
assert all(Path(p).is_file() for p in (
    "report/evidence/e4_data_0.1.json", "report/evidence/e4_data_1.0.json",
    "work/e4_data_0.1_test.jsonl", "work/e4_data_1.0_test.jsonl"))
print("Both E4 conditions restored from Version 3")
run("lab_workflow.py", "results")


## 7. Bonus E5: GRPO with verifiable IoU reward

The reward has six independent CPU tests. GRPO resumes from the trained E3 adapter, generates four completions per image, trains for 60 steps at 5e-6 and saves its own adapter/projector and log. The full projector stays float32; generation uses fp16 autocast to match the Qwen3-VL vision tower.


In [ ]:
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("lab_workflow.py", "grpo")
run("lab_workflow.py", "predict", "--name", "e5")
run("lab_workflow.py", "score", "--name", "e5")
run("lab_workflow.py", "results")


## 8. Visual evidence and return package

Create six paired GT/prediction figures from E3. Export the scored evidence, figures, logs and raw predictions for the final report. The ZIP excludes images, model weights and API tokens.


In [ ]:
best = subprocess.check_output([sys.executable, "lab_workflow.py", "best"], text=True).strip()
run("lab_workflow.py", "figures", "--name", best)
subprocess.run([sys.executable, "scripts/check_submission.py"], check=False)


In [ ]:
import zipfile
archive = Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ("report/evidence/*.json", "report/figures/*.png",
                    "work/*/run.json", "work/*_test.jsonl", "data/splits/meta.json"):
        for path in Path(".").glob(pattern):
            z.write(path, path.as_posix())
print("Download:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
from IPython.display import FileLink, display
display(FileLink(str(archive)))
